# E-commerce Orders Analysis

This project analyzes a multi-table e-commerce dataset using Python, MySQL, SQL, and Power BI.

The goal is to inspect and validate the raw tables, load them into MySQL, perform SQL-based joins and analysis, and build a dashboard to summarize business insights.


In [ ]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine

In [ ]:
project_folder = Path.cwd()
raw_folder = project_folder/"raw_data"
clean_folder = project_folder/"clean_data"
output_folder = project_folder / "output"
sql_folder = project_folder / "sql"


In [ ]:
def audit_df(df, name):
    print(f"Audit: {name}")
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isna().sum())
    print("Duplicate rows:", df.duplicated().sum())
    display(df.head())

In [ ]:
customers = pd.read_csv(raw_folder/"df_Customers.csv")
orderitems= pd.read_csv(raw_folder/"df_OrderItems.csv")
orders = pd.read_csv(raw_folder/"df_Orders.csv")
payments = pd.read_csv(raw_folder/"df_Payments.csv")
products = pd.read_csv(raw_folder/"df_Products.csv")


In [ ]:
audit_df(customers, "customer data")
audit_df(orderitems, "order item data")
audit_df(orders, "order data")
audit_df(payments, "payment data")
audit_df(products, "product data")

## Innitial data quality findings

- The innitial audit indicates that the **customer**, **order item** and **payment** tables contain no missing values or duplicate rows, suggesting good data completeness and no obvious duplication issues at the row level.

- These checks primarily assess completeness and duplication; further validation is required to ensure accuracy, consistency, and relational integrity.

- The **orders** table contains missing values in `order_approved_at` and `order_delivered_timestamp`. These will be further analyzed by `order_status` to determine whether the missing values are expected (e.g. cancelled or unfulfilled orders) or indicate data quality issues.

- The **products** table contains missing values in product category and dimension fields, as well as a high number of duplicate rows. The next step is to verify whether duplicate `product_id` entries have consistent attributes, and then deduplicate the dataset to construct a clean product-level table.

- Next, targeted data quality checks will be performed, including validation of key relationships and consistency across tables, before loading the cleaned datasets into MySQL for SQL analysis.


In [ ]:
print("Customer ID duplicates:" , customers['customer_id'].duplicated().sum())
print("Product ID duplicates:" , products['product_id'].duplicated().sum())
print("Order ID duplicates:" , orders['order_id'].duplicated().sum())

In [ ]:
product_consistency = products.groupby("product_id").nunique()

product_consistency.max()


In [ ]:
products_clean=products.drop_duplicates(subset=['product_id']).reset_index(drop=True)

In [ ]:
audit_df(products_clean, "cleaned product data")

## Product Table Deduplication

The products table contained repeated identical rows. After removing exact duplicates, the table was reduced from 89,316 rows to 27,451 unique product records.

After deduplication, each product record is unique and the table is more suitable to use as a product lookup table for joins.


In [ ]:
missing_category_count = products_clean["product_category_name"].isna().sum()
missing_category_pct = products_clean["product_category_name"].isna().mean() * 100

print("Missing product categories:", missing_category_count)
print("Missing product category percentage:", round(missing_category_pct, 2))

In [ ]:
products_clean['product_category_name']=products_clean['product_category_name'].fillna('unknown')

In [ ]:
audit_df(products_clean, "cleaned product data after category handling")


### Missing Value Handling

The `product_category_name` field contained a small proportion of missing values (~0.51%). 

These were imputed as `"unknown"` to preserve all product records for sql analysis, particularly for categorical aggregations. Given the lack of additional information, it was not feasible to accurately infer the missing categories.

Product dimension fields (`weight`, `length`, `height`, `width`) contained only 2 missing records. These were retained as null values (`NaN`), as imputing physical measurements would introduce inaccuracies, and the negligible proportion does not materially impact analysis.

## Order Data Investigation

The orders dataset contains no duplicate rows. 

The `order_delivered_timestamp` field has 1,889 missing values, which require validation against `order_status` to determine whether they are expected (e.g. cancelled or unfulfilled orders).

The `order_approved_at` field has 9 missing values (~0.01%) and was retained as null (`NaN`) due to its negligible impact.


## Exploratory Observations

Inconsistencies were identified where cancelled orders contain delivery timestamps, violating expected business logic.

These were validated programmatically in pandas, and a flag was created to identify such records without modifying the original data.

In [ ]:
orders["invalid_delivery_flag"] = (
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_timestamp"].notna())
)

In [ ]:
orders[orders["invalid_delivery_flag"]]


In [ ]:
orders.groupby("order_status")["order_delivered_timestamp"].apply(lambda x: x.isna().sum())

In [ ]:
orders['missing_delivery_timestamp']=(
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_timestamp"].isna())
)

In [ ]:
orders[orders['missing_delivery_timestamp']]

## Missing delivery timestamps analyzed against `order_status`

The majority of missing values correspond to non-delivered statuses (e.g. cancelled, processing, shipped), which is expected.

However, 6 records were identified and flagged where orders marked as "delivered" lack delivery timestamps, indicating data inconsistencies.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_timestamp",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


In [ ]:
audit_df(orders,'date standardisation of order data')

Date columns were converted to datetime format using `pd.to_datetime()`. The missing value counts remained unchanged after conversion, indicating that no additional date values failed to parse during standardisation.

In [ ]:
payments.dtypes

In [ ]:
orderitems.dtypes

In [ ]:
payments[["payment_value", "payment_installments"]].describe()

In [ ]:
orderitems[["price", "shipping_charges"]].describe()

## Numeric Field Validation

The key numeric fields in the `payments` and `orderitems` tables were reviewed before loading into MySQL.

The fields `payment_value`, `payment_installments`, `price`, and `shipping_charges` are stored as numeric data types, making them suitable for calculations.

Summary statistics showed no negative values in these fields. Zero values were present in some payment and shipping fields, which may represent free shipping, voucher usage, or special order/payment cases. These values were retained for analysis because there is not enough evidence to treat them as invalid.


## Final Validation Before SQL Upload

The key data quality checks have been completed before loading the tables into MySQL.

- Customer, order, order item, and payment tables were checked for missing values and duplicate rows.
- Key identifiers such as `customer_id`, `order_id`, and `product_id` were reviewed for expected uniqueness.
- The products table was deduplicated to create a clean product lookup table.
- Missing product categories were labeled as `unknown`.
- Missing product dimension values were left unchanged because there was no reliable source to infer them.
- Order timestamp issues were reviewed against `order_status`, and suspicious delivery records were flagged.
- Numeric fields such as `price`, `shipping_charges`, and `payment_value` were confirmed to be numeric and suitable for calculations.

Based on these checks, the cleaned tables are ready to be loaded into MySQL for relationship checks and SQL-based analysis.


In [ ]:
cleaned_tables= {
    "customers_clean":customers,
    "order_items_clean":orderitems,
    "orders_clean":orders,
    "payments_clean":payments,
    "products_clean":products
}
for table_name,df in cleaned_tables.items():
    clean_folder_path=clean_folder / f"{table_name}.csv"
    df.to_csv(clean_folder_path, index=False)
    print(f"Saved {table_name}: {len(df)} rows -> {clean_folder_path}")

The cleaned tables were exported as CSV files to the `clean_data` folder. CSV format was used because these files are intended for loading into MySQL and future analysis workflows.


## Required Packages

If needed, install the required packages with:

`%pip install pandas openpyxl sqlalchemy pymysql`


In [ ]:
mysql_user = "root"
mysql_password = getpass("Enter MySQL password: ")
mysql_host = "localhost"
mysql_port = 3306
mysql_database = "ecom_analysis"

engine = create_engine(
    f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_database}"
)

In [ ]:
sql_tables = {
    "customers": customers,
    "orders": orders,
    "order_items": orderitems,
    "payments": payments,
    "products": products_clean
}

for table_name, df in sql_tables.items():
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )
    
    row_count = pd.read_sql(
        f"SELECT COUNT(*) AS total_rows FROM {table_name}",
        engine
    )
    
    print(f"Uploaded table: {table_name}")
    display(row_count)


## MySQL Upload Validation

The MySQL row counts matched the pandas row counts after upload.

The `customers`, `orders`, `order_items`, and `payments` tables each uploaded successfully with 89,316 rows.

The cleaned `products` table uploaded successfully with 27,451 rows after removing duplicate product records.

The tables are now ready for SQL relationship checks and business analysis.
